In [1]:
from typing import Any
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser


load_dotenv()

C:\2026-Projects\Document_Intelligent_Hub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PROJECT_ROOT = Path.cwd()

DATA_DIR= PROJECT_ROOT / "data"
VECTOR_STORE_DIR = DATA_DIR / "vector_store" / "chroma"


In [3]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_function = HuggingFaceEmbeddings(
    model_name = EMBEDDING_MODEL_NAME,
    model_kwargs = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7606.94it/s]


In [4]:
COLLECTION_NAME = "enterprise_documents"

vector_store = Chroma(
    collection_name = COLLECTION_NAME,
    embedding_function = embedding_function,
    persist_directory = str(VECTOR_STORE_DIR),
)

collection_count = vector_store._collection.count()

print(f"Loaded collection: {COLLECTION_NAME}")
print(f"Document/ chunks in collection: {collection_count}")

Loaded collection: enterprise_documents
Document/ chunks in collection: 5


In [5]:
llm = ChatOllama(
    model= "qwen3:8b",
    temperature=0
)


In [6]:
def normalize_user_role(user_role: str) -> str:
    """Normalize user role to stored metadata format."""
    role = user_role.strip().lower().replace("-", "_").replace(" ", "_")

    role_map = {
        "admin": "Admin",
        "compliance_analyst": "Compliance Analyst",
        "viewer": "Viewer",
    }
    return role_map.get(role, user_role)

In [7]:
def retrieve_documents_with_role_access(
        query: str,
        user_role: str,
        k: int = 5,
) -> list[tuple[Document, float]]:
    """Retrieve documents with role-based access control."""
    normalized_role = normalize_user_role(user_role)

    if normalized_role == "Admin":
        results = vector_store.similarity_search_with_relevance_scores(
            query = query,
            k = k,
        )
    else:
        results = vector_store.similarity_search_with_relevance_scores(
            query = query,
            k = k,
            filter= {"access_role": normalized_role},
        )
    return results

In [ ]:
retrieved_results = retrieve_documents_with_role_access(
    query = "What is the purpose of the company?",
    user_role = "Compliance Analyst",
    k=5,
)

print(f"Retrieved {len(retrieved_results)} result(s) for query:")

In [9]:
for index, (document, score) in enumerate(retrieved_results, start=1):
    print("=" * 100)
    print(f"Result {index}")
    print(f"Score: {score:.4f}")
    print("Source:", document.metadata.get("source"))
    print("Page:", document.metadata.get("page_number"))
    print("Chunk ID:", document.metadata.get("chunk_id"))
    print("-" * 100)
    print(document.page_content[:700])

Result 1
Score: -0.0946
Source: Sample Compliance Manual.pdf
Page: 1
Chunk ID: 40b1525f-5997-4905-993a-07e5013123b1
----------------------------------------------------------------------------------------------------
• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employees. 
• Certification records must be retained for 5 years.
Result 2
Score: -0.1394
Source: Sample Compliance Manual.pdf
Page: 1
Chunk ID: 2e963208-afd2-4d35-b36c-0c92fe541fbc
----------------------------------------------------------------------------------------------------
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Com

In [17]:
def format_documents_as_context(
        retrieved_results: list[tuple[Document, float]],
) -> str:
    """Format retrieved documents as context for the LLM."""
    context_parts =[]

    for index, (document, score) in enumerate(retrieved_results, start=1):
        source = document.metadata.get("source", "Unknown Source")
        page_number = document.metadata.get("page_number", "Unknown Page")
        chunk_id = document.metadata.get("chunk_id", "Unknown Chunk")

    context_part = f"""
[Source {index}] Document: {source} Page: {page_number} Chunk ID: {chunk_id} Relevance Score: {score:.4f} Content:{document.page_content}
""".strip()

    context_parts.append(context_part)

    return "\n\n".join(context_parts)

In [18]:
def extract_citations(
        retrieved_results: list[tuple[Document, float]],
) -> list[dict[str, Any]]:
    """Extract unique citations from retrieved documents."""
    citations = []
    seen = set()

    for index, (document, score) in enumerate(retrieved_results, start=1):
        source = document.metadata.get("source", "Unknown Source")
        page_number = document.metadata.get("page_number", "N/A")
        chunk_id = document.metadata.get("chunk_id", "N/A")

        citation_key = (source, page_number, chunk_id)

        if citation_key in seen:
            continue

            seen.add(citation_key)

            citations.append(
                {
                    "citation_id": len(citations) + 1,
                    "source": source,
                    "page_number": page_number,
                    "chunk_id": chunk_id,
                    "score": float(score),
                }
            )
    return citations

In [20]:
test_citations = extract_citations(retrieved_results)
print(f"\nExtracted {len(test_citations)} unique citations:")
for citation in test_citations:
    print(f"  [{citation['citation_id']}] {citation['source']} - Page {citation['page_number']} (Score: {citation['score']:.4f})")

context = format_documents_as_context(retrieved_results)
print("\nFormatted Context (first 1000 chars):")
print(context[:1000])


Extracted 0 unique citations:

Formatted Context (first 1000 chars):
[Source 5] Document: Sample Compliance Manual.pdf Page: 2 Chunk ID: ccdd3435-52ed-4582-8713-cf0a7381e819 Relevance Score: -0.2383 Content:• High-risk findings must be escalated to the Risk Committee within 7 business 
days. 
Business Continuity 
• Continuity plans must be tested annually. 
• Disaster recovery drills must be documented and reviewed. 
Vendor Compliance 
• Vendors must provide proof of compliance with ISO 27001 standards.


In [21]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
             """
        You are an enterprise compliance knowledge assistant.

        Your job is to answer user questions using ONLY the provided context.

        Rules:
        1. Use only the provided context.
        2. If the answer is not in the context, say "I could not find that information in the available document.
        3. Do not invent policies, thresholds, dates, or requirements.
        4. Include citations using the provided source numbers, example: [Source 1]
        5. Be concise, accurate, and compliance-focused.
        6. If there are multiple relevant sources, cite all relevant sources.

        Context:
        {context}
        """.strip(),
        ),
        (
        "human",
            "Question: {question}",
        ),
    ]
)

In [13]:
qa_chain = qa_prompt | llm | StrOutputParser()

In [22]:
question = "Tell me about Anti-Money Laundering Obligations "

retrieved_results = retrieve_documents_with_role_access(
    query=question,
    user_role="Admin",
    k=5,
)

context = format_documents_as_context(retrieved_results)
citations = extract_citations(retrieved_results)
print(f"Question: {question}")
print("\nCitations Found:")
for citation in citations:
    print(f"  [{citation['citation_id']}] {citation['source']} - Page {citation['page_number']} (Score: {citation['score']:.4f})")

print("\nGenerating Answer...")
answer = qa_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)
print("\nAnswer:")
print(answer)

C:\Users\benne\AppData\Local\Temp\ipykernel_9116\259747806.py:9: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_0', metadata={'chunk_size': 991, 'source': 'Sample Compliance Manual.pdf', 'page_number': 1, 'producer': 'Microsoft® Word for Microsoft 365', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'creator': 'Microsoft® Word for Microsoft 365', 'department': 'Compliance', 'page_label': '1', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'access_role': 'Compliance Analyst', 'total_pages': 3, 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'page': 0, 'file_type': 'pdf', 'chunk_index': 0, 'moddate': '2026-06-20T21:35:22+02:00', 'part_index': 0}, page_content='Sample Compliance Manual \n \nPage 1 — Introduction & Scope \nThis Compliance 

Question: Tell me about Anti-Money Laundering Obligations 

Citations Found:

Generating Answer...
I could not find that information in the available document. The provided context does not include details about Anti-Money Laundering (AML) obligations [Source 5].
